# Milestone: Static Generator + JAX AD Pipeline

This notebook verifies the current milestone:
- deterministic structured mesh generation
- NumPy/JAX forward parity
- compile-once, run-many JAX optimization timing

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp

from gmshjax.mesh.topology import unit_square_tri_mesh, unit_square_quad_mesh, unit_cube_tet_mesh
from gmshjax.numpy_impl import unit_square_tri_mesh as np_unit_square_tri_mesh
from gmshjax.ad.pipeline import build_model_parametric_quality_value_and_grad, default_param_vector
from gmshjax.mesh.factory import make_unit_square_model

In [ ]:
# Deterministic static generation
topo_a, pts_a = unit_square_tri_mesh(32, 24)
topo_b, pts_b = unit_square_tri_mesh(32, 24)
print('tri_elements_equal:', np.array_equal(np.asarray(topo_a.elements), np.asarray(topo_b.elements)))
print('tri_points_equal:', np.allclose(np.asarray(pts_a), np.asarray(pts_b)))

topo_q, pts_q = unit_square_quad_mesh(32, 24)
topo_t, pts_t = unit_cube_tet_mesh(10, 10, 8)
print('quad shape:', topo_q.elements.shape, pts_q.shape)
print('tet shape:', topo_t.elements.shape, pts_t.shape)

In [ ]:
# NumPy/JAX topology parity for triangles
topo_jax, _ = unit_square_tri_mesh(20, 15)
topo_np, _ = np_unit_square_tri_mesh(20, 15, dtype=np.float32)
print('elements parity:', np.array_equal(np.asarray(topo_jax.elements), topo_np.elements))
print('edges parity:', np.array_equal(np.asarray(topo_jax.edges), topo_np.edges))

In [ ]:
# JAX compile-once benchmark
model = make_unit_square_model(64, 64)
value_and_grad = build_model_parametric_quality_value_and_grad(model)
theta = default_param_vector()

t0 = time.perf_counter()
value, grad = value_and_grad(theta)
_ = jax.block_until_ready((value, grad))
t1 = time.perf_counter()

n_steps = 100
t2 = time.perf_counter()
for _ in range(n_steps):
    value, grad = value_and_grad(theta)
    theta = theta - 0.05 * grad
_ = jax.block_until_ready((value, grad))
t3 = time.perf_counter()

print('first_call_ms:', (t1 - t0) * 1e3)
print('steady_state_ms_per_step:', ((t3 - t2) / n_steps) * 1e3)
print('final_value:', float(value))
print('final_grad_norm:', float(jnp.linalg.norm(grad)))